# English Listening Video Generator (方案 B — Grouped Multi-Character)

Generates a ~13min English listening practice video with:
- LLM script (SenseNova DeepSeek V4 Flash, 18 dialogue lines + IPA + 繁中)
- 3 frontier character/scene images (3D cartoon style)
- Grouped Seedance2 video clips (方案 B: merges consecutive lines, both char refs)
- Kokoro TTS (English) + edge-tts (Chinese) + loudnorm
- FFmpeg + Pillow composition with bilingual subtitles

**Colab GPU recommended** for Kokoro TTS speed.

## Setup
1. Run Cell 1 to install dependencies (~3min)
2. Run Cell 2 to clone scripts from GitHub (automatic)
3. Fill in your tokens in Cell 3 (MCP token + LLM API key)
4. Run Cell 4 to generate the video

## Cell 1: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg git

# Install Python dependencies
!pip install -q kokoro soundfile torch edge-tts Pillow opencc-python-reimplemented cn2an pypinyin ordered_set jieba

# Install CJK fonts for Pillow rendering
!apt-get install -y -qq fonts-noto-cjk fonts-dejavu-core

# Create working directory
import os
os.makedirs('/content/output', exist_ok=True)

print('Dependencies installed.')
print(f'FFmpeg: {os.popen("ffmpeg -version 2>/dev/null | head -1").read().strip()}')

## Cell 2: Clone Scripts from GitHub

Automatically downloads all scripts from the GitHub repository.

In [ ]:
import os

repo_url = 'https://github.com/collinsgraciano/colab-listening-b.git'
clone_dir = '/content/colab-listening-b'

if os.path.exists(clone_dir):
    !cd {clone_dir} && git pull -q
    print(f'Updated existing clone: {clone_dir}')
else:
    !git clone -q {repo_url} {clone_dir}
    print(f'Cloned to: {clone_dir}')

# Verify all required scripts are present
required = ['mcp_client.py', 'llm_client.py', 'tts_engine.py', 'timeline.py',
            'grouping_b.py', 'video_compose.py', 'pipeline.py', 'topic_manager.py',
            'topics.json']
missing = [f for f in required if not os.path.exists(os.path.join(clone_dir, f))]
if missing:
    print(f'Missing files: {missing}')
else:
    print('All scripts found!')

## Cell 3: Set Tokens

### MCP Token

Get your TJGenerators MCP OAuth token from your local machine:
```python
# On your Windows machine, run:
import json
tokens = json.load(open(r'C:\Users\Administrator\.codely-cli\mcp-oauth-tokens.json'))
token = next(t['token']['accessToken'] for t in tokens if t['serverName'] == 'TJGenerators')
print(token)
```

### LLM API

Default uses SenseNova DeepSeek V4 Flash. To use a different OpenAI-compatible API, change the settings below.

In [ ]:
import os

# === MCP Token ===
MCP_TOKEN = 'PASTE_YOUR_MCP_TOKEN_HERE'

# === LLM API Configuration ===
# Default: SenseNova DeepSeek V4 Flash (OpenAI-compatible)
# Replace with your own API key if needed
LLM_API_KEY = 'sk-8Tr86c17YvA5jBEoem2uYYAQGXGzmpDU'
LLM_BASE_URL = 'https://token.sensenova.cn/v1'
LLM_MODEL = 'deepseek-v4-flash'

# Set environment variables (inherited by pipeline subprocess)
os.environ['SENSENOVA_API_KEY'] = LLM_API_KEY
os.environ['SENSENOVA_BASE'] = LLM_BASE_URL
os.environ['SENSENOVA_MODEL'] = LLM_MODEL

# Verify
if MCP_TOKEN == 'PASTE_YOUR_MCP_TOKEN_HERE':
    print('WARNING: Please paste your MCP token above!')
else:
    print(f'MCP token set ({len(MCP_TOKEN)} chars)')
print(f'LLM API: {LLM_BASE_URL} / {LLM_MODEL}')

## Cell 4: Generate Video

Set the topic and CEFR level, then run. The pipeline takes ~15-25 minutes.

In [ ]:
import subprocess, sys, os

TOPIC = ''  # Leave empty to pick randomly from topics.json
CEFR = 'A2'  # A1, A2, B1, B2, C1, C2
OUTPUT_DIR = '/content/output'
SCRIPTS_DIR = '/content/colab-listening-b'

cmd = [
    sys.executable, 'pipeline.py',
    '--cefr', CEFR,
    '--output', OUTPUT_DIR,
    '--mcp-token', MCP_TOKEN,
    '--topics-file', os.path.join(SCRIPTS_DIR, 'topics.json'),
    '--used-topics-file', '/content/output/used_topics.json',
]
if TOPIC:
    cmd.extend(['--topic', TOPIC])

result = subprocess.run(cmd, cwd=SCRIPTS_DIR)
if result.returncode != 0:
    print(f'Pipeline failed with exit code {result.returncode}')

## Cell 5: Download Result

In [ ]:
from google.colab import files
import os

video_path = '/content/output/videos/final_video.mp4'
if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024*1024)
    print(f'Video: {video_path} ({size_mb:.1f}MB)')
    files.download(video_path)
else:
    print('Video not found. Check pipeline output above for errors.')

## Cell 6: Preview Video (optional)

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

video_path = '/content/output/videos/final_video.mp4'
if os.path.exists(video_path):
    with open(video_path, 'rb') as f:
        video_b64 = b64encode(f.read()).decode()
    html = f'''
    <video width="640" height="360" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''
    display(HTML(html))
else:
    print('Video not found.')